# Golay Encoder

In [23]:
%load_ext autoreload
%autoreload 1
%aimport classes.GaloisField
%aimport classes.GolayDecoder

import numpy as np
import os

from classes.GaloisField import *
from classes.GaloisPoly  import *
from classes.GolayEncoder import GolayEncoder

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
gf            = GaloisField(1, 0b11)
encoder_model = GolayEncoder()
k             = encoder_model._k
n             = encoder_model._n

Field Closed Succesfully!, 1 Non-Zero Elements


## Verifify code properties

In [25]:
G2412G  = np.hstack((np.eye(k), encoder_model._G2412B)).astype(np.uint8)
G2412H  = np.hstack((encoder_model._G2412B.T, np.eye(n-k))).astype(np.uint8)

# test code properties
g_ht    = gf.mat_mul(G2412G, G2412H.T)
b_2     = gf.mat_mul(encoder_model._G2412B, encoder_model._G2412B)
print(f"Property: G*HT = 0")
print(f"{32*'='}")
print(g_ht)
print()
print(f"Property: B*B = I")
print(f"{32*'='}")
print(b_2)

Property: G*HT = 0
[[0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]]

Property: B*B = I
[[1 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 1]]


In [26]:
print(G2412G)

[[1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 1 0 0 0 1 1 1 1]
 [0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 1 1 0 0 1 1 1]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 1 0 1 1 1]
 [0 0 0 1 0 0 0 0 0 0 0 0 1 0 1 1 1 1 1 0 0 0 1 0]
 [0 0 0 0 1 0 0 0 0 0 0 0 1 1 0 1 1 1 0 1 0 0 0 1]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 1 1 1 1 1 0 0 1 1 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 1 0 0 1 1 1 1 0 1]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 1 0 1 1 1 1 1 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 1 1 1 1 0 1 1]
 [0 0 0 0 0 0 0 0 0 1 0 0 1 1 1 0 0 1 1 1 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 1 1 1 1 0 0 0 1 1 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 0 1 0 1 0 1 0 0 1]]


In [27]:
encoder_output = None

words = []

for w in range(2**k):
    w  = gf.do_unpack(w, bit_width= k)
    words.append(w)
    cw = encoder_model.encode(w)
    encoder_output = cw if encoder_output is None else np.vstack((encoder_output, cw))

#encoder_output
# n_codewords = len(encoder_output)
# encoder_output[3838]  

In [28]:
import itertools
import math
import numpy as np

# ============================================================
# Select codeword
# ============================================================

cw = encoder_output[3838]

# Convert to a string of bits
cw_str = ''.join(map(str, cw))


# ============================================================
# Output files
# ============================================================

filenames = [
    "outputs/error_injection/all_errors_cw.txt",
    "../implem/error_injection/error_injection_testing/all_errors_cw.txt"
]


# ============================================================
# Generate all error masks with weight 0, 1, 2, 3 and 4
# ============================================================

n = len(cw)

n_cases = sum(math.comb(n, w) for w in range(5))


for filename in filenames:

    with open(filename, "w") as f:

        for weight in range(5):

            # All combinations of 'weight' bit positions
            for positions in itertools.combinations(range(n), weight):

                # Create error mask
                error_mask = np.zeros(n, dtype=int)

                for pos in positions:
                    error_mask[pos] = 1

                # Inject error
                cw_error = np.bitwise_xor(cw, error_mask)

                # Convert to strings
                error_mask_str = ''.join(map(str, error_mask))
                cw_error_str = ''.join(map(str, cw_error))

                # Write:
                # encoder_output   error_mask   codeword_with_error
                f.write(
                    f"{cw_str} {error_mask_str} {cw_error_str}\n"
                )

    print(f"Generated {n_cases} cases")
    print(f"Output: {filename}")

Generated 12951 cases
Output: outputs/error_injection/all_errors_cw.txt
Generated 12951 cases
Output: ../implem/error_injection/error_injection_testing/all_errors_cw.txt


In [29]:
# Paths de salida
filename_encoder_vectors_1 = "outputs/encoder/encoder_top_testing/encoder_output.txt"
filename_encoder_vectors_2 = "../implem/golay_encoder/golay_encoder_testing/encoder_output.txt"


# Crear directorios si no existen
os.makedirs(os.path.dirname(filename_encoder_vectors_1), exist_ok=True)
os.makedirs(os.path.dirname(filename_encoder_vectors_2), exist_ok=True)


# Escribir en ambos archivos
with open(filename_encoder_vectors_1, "w") as f1, \
     open(filename_encoder_vectors_2, "w") as f2:

    for w, cw in zip(words, encoder_output):

        # Convertir los vectores de bits a strings
        w_str  = ''.join(map(str, w))
        cw_str = ''.join(map(str, cw))

        # Una línea: 12 bits + espacio + 24 bits
        line = f"{w_str} {cw_str}\n"

        f1.write(line)
        f2.write(line)


print(f"Encoder vectors written to:")
print(f"  {filename_encoder_vectors_1}")
print(f"  {filename_encoder_vectors_2}")

Encoder vectors written to:
  outputs/encoder/encoder_top_testing/encoder_output.txt
  ../implem/golay_encoder/golay_encoder_testing/encoder_output.txt


## Get weight distribution

In [30]:
# get weight distribution, match with golay 24,12 distribution
weights = np.full(n+1, -1).astype(np.int32)
for cw in encoder_output:
    w = gf.hamming_weight(cw)
    weights[w] = 1 if weights[w] == -1 else weights[w]+1

weights

print(f"Golay 24,12 Weight Distribution:")
print(f"{32*'='}")
for i, w in enumerate(weights):
    if w != -1:
        print(f"W{i}\t: {w}")

Golay 24,12 Weight Distribution:
W0	: 1
W8	: 759
W12	: 2576
W16	: 759
W24	: 1
